# EC & IM: Bayesian Hierarchical Analysis

A narrative walkthrough of the analysis. For the full pipeline see
`src/run_analysis.py`; this notebook explains the reasoning step by step.

**Setup:** install requirements and place the two CSVs in `../data/`
(see `data/README.md`).

In [ ]:
import sys
sys.path.append("..")
from pathlib import Path
from src import data as data_mod
from src import models, diagnostics as diag, plots
DATA_DIR = Path("../data")
FIG_DIR = Path("../figures"); FIG_DIR.mkdir(exist_ok=True)

## 1. Load and clean the data

Merge the Social Capital Atlas (EC) and Opportunity Atlas (IM) frames on a FIPS
key, then drop rows with missing, negative, or out-of-range values and the
"average" summary rows. IM percentiles above 1 come from privacy noise added by
the data providers.

In [ ]:
df = data_mod.load_clean(DATA_DIR)
print(f"{len(df)} counties across {df['state'].nunique()} states")
df.head()

## 2. Motivating plots

A positive EC-IM trend, and clear variation in baseline IM across states - the
latter motivates a hierarchical model.

In [ ]:
plots.scatter_ec_im(df, FIG_DIR / "ec_vs_im.png")
plots.state_boxplot(df, FIG_DIR / "im_by_state.png")

## 3. Question 1 - Complete pooling

A single regression of IM on EC, ignoring state structure. Weakly informative
priors; with ~3,000 counties the likelihood dominates.

In [ ]:
cp_model, cp_idata = models.fit_complete_pooling(df)
diag.summary(cp_idata, var_names=["Intercept", "ec_county", "sigma"])

In [ ]:
print("Converged:", diag.converged(cp_idata))
lo, hi = diag.credible_interval(cp_idata, "ec_county")
print(f"95% CI for EC slope: ({lo:.4f}, {hi:.4f})")
plots.ppc(cp_model, cp_idata, FIG_DIR / "ppc_complete_pooling.png")

The EC slope is positive and excludes zero. But the posterior predictive
check shows the pooled model doesn't capture the data's structure. That
motivates Question 2.

## 4. Question 2 - Hierarchical (varying intercept by state)

Each state gets its own baseline IM offset, partially pooled toward the global
intercept so small states borrow strength.

In [ ]:
h_model, h_idata = models.fit_hierarchical(df)
print("Converged:", diag.converged(h_idata))
lo, hi = diag.credible_interval(h_idata, "ec_county")
print(f"95% CI for EC slope: ({lo:.4f}, {hi:.4f})")

In [ ]:
var = models.variance_decomposition(h_idata)
print(f"Between-state variance: {var['between_state']*100:.1f}%")
print(f"Within-state variance:  {var['within_state']*100:.1f}%")
plots.ppc(h_model, h_idata, FIG_DIR / "ppc_hierarchical.png")

## 5. Conclusions

- The EC-IM relationship is **positive** under both models.
- About **60% of IM variation is between states**, justifying the hierarchical
  model.
- Modeling state structure **shrinks the EC slope** but does not change its
  sign; the hierarchical posterior predictive check fits better.